# **Dundee Housing Sites - Manual Data Cleaning**
**Overview**

This analysis uses the housingSites.csv dataset to clean key text fields, explore tenure types, assess site capacity, and examine projected housing delivery between 2024 and 2027. The objective is to understand how tenure type and site status relate to projected build volumes.

Attached files:

*   Raw dataset: housingSites.csv
*   Cleaned dataset: housing_sites_manual_cleaned.csv
*   Python notebook: HousingSites_Manual_DataCleaning.ipynb

**Data Preparation and Cleaning**

The dataset contains 183 housing sites and 40 columns. To focus on answering the questions, a subset of relevant columns was used to create a new DataFrame, such as Site address, Owner/Developer, Tenure type, Site capacity, Site status, and Yearly projected build columns.
The Owner/Developer and Site address columns were cleaned and standardised by:

*   Removing leading and trailing spaces
*   Replacing multiple internal spaces with a single space
*   Applying consistent title-case formatting

These columns were converted to string data type to ensure consistency and improve data quality.

**Tenure Types and Site Capacity**

Eight unique tenure types were identified in the dataset. Total site capacity was calculated for each tenure type and stored in a dictionary. The results show that Private tenure sites dominate overall site capacity, followed by Registered Social Landlord (RSL) and TBC categories. Other tenure types, such as Local Authority and Housing Association, contribute relatively small proportions of total capacity.

**Projected Builds for Under-Construction Sites (2024–2027)**

Sites with a status of “Under Construction” were used to represent the “CONS” category. This subset contains 62 sites. Total projected builds were calculated across the four-year period from 2024/25 to 2027/28.
The analysis shows a total of 1,412 projected housing units during this period, with the highest contributions occurring in the earlier years. This indicates that under-construction sites play a major role in near-term housing delivery.

**Relationship Between Tenure Type and Projected Build Volumes**

The results indicate a clear relationship between tenure type and projected build volumes. Private tenure sites account for the largest share of projected builds, reflecting their dominance in overall site capacity. RSL sites also contribute significantly, while mixed and other tenure categories provide smaller but notable contributions. This suggests that housing delivery between 2024 and 2027 is largely driven by private-led developments, with social and mixed tenures playing a supporting role.

**Conclusion**

This analysis demonstrates that data cleaning is essential for reliable results and that tenure type is a strong indicator of both site capacity and projected housing delivery. Under-construction sites, particularly those with private tenure, are the primary contributors to short-term housing supply.

In [ ]:
#import python libraries
import pandas as pd

In [ ]:
#Read csv file
housing_sites_df = pd.read_csv('housingSites.csv')

#Expolre the DataFrame
print('1. First five rows of DataFrame: ')
print(housing_sites_df.head())

print('\n2. Name of columns: ')
print(housing_sites_df.columns)
print(housing_sites_df.columns.to_list())

print('\n3. Summary info of DataFrame: ')
print(housing_sites_df.info())

print('\n4. Number of rows and columns: ')
print(housing_sites_df.shape)

print('\n5. Missing values of DataFrame: ')
print(housing_sites_df.isna().sum())


1. First five rows of DataFrame: 
   OBJECTID Site reference LDP2 reference         Year site added  \
0         1         200321            H13        04/01/2003 00:00   
1         2         200611            NaN        06/05/2007 00:00   
2         3         200347            NaN  12/17/2013 12:00:00 AM   
3         4         201702            NaN        07/01/2016 00:00   
4         5         200909            H11        04/01/2009 00:00   

   Site area (ha)                           Site address  Easting  Northing  \
0        1.249780                   QUEEN VICTORIA WORKS   339248    730423   
1        1.677644  RIVERSIDE DRIVE, FORMER HOMEBASE SITE   339577    729422   
2        1.006731             MONIFIETH ROAD, ARMITSTEAD   347412    731148   
3        0.459596                       GRAY STREET, 44    337271    731747   
4        1.277837  EAST SCHOOL ROAD, FORMER DOWNFIELD PS   338919    733132   

     Site type         Site status  ... Year 28/29 Year 29/30 Year 30/31  \


In [ ]:
#Create a subset of my housing_sites_df called submission_1_df - to contain only that columns which are required for analysis

submission_1_df = housing_sites_df[['Site address', 'Owner/Developer', 'Tenure type', 'Site capacity',
  'Site status', 'Year 24/25', 'Year 25/26', 'Year 26/27', 'Year 27/28']].copy()


In [ ]:
#Exploratory data analysis of submission_1_df

print('1. Summary info of submission_1_df: ')
print(submission_1_df.info())

print('\n2. First five rows of submission_1_df: ')
print(submission_1_df.head())

print('\n3. Number of rows and columns: ')
print(submission_1_df.shape)

print('\n4. Missing values of submission_1_df: ')
print(submission_1_df.isna().sum())

print('\n5. Name of columns: ')
print(submission_1_df.columns)
print(submission_1_df.columns.to_list())

print('\n6. Unique values of Tenure type column: ')
print(submission_1_df['Tenure type'].unique())

print('\n7. Unique values of Site status column: ')
print(submission_1_df['Site status'].unique())

#Change object data type of Tenure type and Site status columns to category
submission_1_df['Tenure type'] = submission_1_df['Tenure type'].astype('category')
submission_1_df['Site status'] = submission_1_df['Site status'].astype('category')

1. Summary info of submission_1_df: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 183 entries, 0 to 182
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Site address     183 non-null    object
 1   Owner/Developer  183 non-null    object
 2   Tenure type      183 non-null    object
 3   Site capacity    183 non-null    int64 
 4   Site status      183 non-null    object
 5   Year 24/25       183 non-null    int64 
 6   Year 25/26       183 non-null    int64 
 7   Year 26/27       183 non-null    int64 
 8   Year 27/28       183 non-null    int64 
dtypes: int64(5), object(4)
memory usage: 13.0+ KB
None

2. First five rows of submission_1_df: 
                            Site address       Owner/Developer Tenure type  \
0                   QUEEN VICTORIA WORKS               PRIVATE     Private   
1  RIVERSIDE DRIVE, FORMER HOMEBASE SITE  H & H PROPERTIES LTD     Private   
2             MONIFIETH ROAD, ARM

In [ ]:
#Clean and standardise the Owner/Developer and Site Name columns
#by removing extra spaces and applying consistent capitalization
#and I want to change the object datatype to string as well

#clean function - it is not recommended, if the function in the same notebook as my data
def clean_text(text):
  if isinstance(text, str):
    text = text.strip().title()
    return text

#clean text column function in pandas
def clean_text_column(series):
    return series.astype('string').str.strip().str.replace(r'\s+', ' ', regex=True).str.title()

#Use the two different method on Owner/Developer - they work differently
'''
housing_sites_df['Owner/Developer'] = housing_sites_df['Owner/Developer'].apply(clean_text)
print(housing_sites_df['Owner/Developer'].dtype)
'''

submission_1_df['Owner/Developer'] = clean_text_column(submission_1_df['Owner/Developer'])
submission_1_df['Site address'] = clean_text_column(submission_1_df['Site address'])
print("Data type of 'Owner/Developer' column: ", submission_1_df['Owner/Developer'].dtype)
print("Data type of 'Site address' column: ", submission_1_df['Site address'].dtype)


#Display summary information about DataFrame
print('Summary info of submission_1_df: ')
print(submission_1_df.info())

#Export Clean Dataset
submission_1_df.to_csv('submission2_week3_clean.csv', index = False)

Data type of 'Owner/Developer' column:  string
Data type of 'Site address' column:  string
Summary info of submission_1_df: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 183 entries, 0 to 182
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   Site address     183 non-null    string  
 1   Owner/Developer  183 non-null    string  
 2   Tenure type      183 non-null    category
 3   Site capacity    183 non-null    int64   
 4   Site status      183 non-null    category
 5   Year 24/25       183 non-null    int64   
 6   Year 25/26       183 non-null    int64   
 7   Year 26/27       183 non-null    int64   
 8   Year 27/28       183 non-null    int64   
dtypes: category(2), int64(5), string(2)
memory usage: 11.1 KB
None


In [ ]:
#Identify all unique Tenure Types and calculate the total Site Capacity for each category
#store your results in a dictionary

site_capacity_by_tenure = (submission_1_df \
  .groupby('Tenure type')['Site capacity'] \
  .sum() \
  .sort_values(ascending=False).to_dict())
print(site_capacity_by_tenure)

#Alternative solution using for loop - recommended by course
#Unique tenure types
tenure_types = set(housing_sites_df['Tenure type'])
print("Tenure Types: ", tenure_types)
#for loop
capacity_by_tenure = {}
for tenure in tenure_types:
  total = housing_sites_df[housing_sites_df['Tenure type'] == tenure]['Site capacity'].sum()
  capacity_by_tenure[tenure] = total
print("Capacity by tenure: ", capacity_by_tenure)

{'Private': 3503, 'RSL': 840, 'TBC': 419, 'Mixed': 343, 'DCC/RSL': 162, 'Local Authority': 43, 'Private/RSL': 43, 'Housing Association': 18}
Tenure Types:  {'Private/RSL', 'Private', 'TBC', 'Local Authority', 'Housing Association', 'DCC/RSL', 'Mixed', 'RSL'}
Capacity by tenure:  {'Private/RSL': np.int64(43), 'Private': np.int64(3503), 'TBC': np.int64(419), 'Local Authority': np.int64(43), 'Housing Association': np.int64(18), 'DCC/RSL': np.int64(162), 'Mixed': np.int64(343), 'RSL': np.int64(840)}


/tmp/ipython-input-4024917424.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby('Tenure type')['Site capacity'] \


In [ ]:
#Filter the dataset to include only sites with status "CONS" and calculate the total projected builds for 2024- 2027
#I assume that the Under construction category means "CONS" - So I will filter for this category
print(submission_1_df['Site status'].unique().tolist())
print(submission_1_df['Site status'].value_counts())


submission_1_df_filtered = submission_1_df[submission_1_df['Site status'] == 'Under Construction']
print(submission_1_df_filtered.shape)

year_columns = ['Year 24/25', 'Year 25/26', 'Year 26/27', 'Year 27/28']
total_proj_builds = submission_1_df_filtered[year_columns].sum()
print('Total Projected Builds yearly between 2024-2027: ', total_proj_builds)
total_proj_builds_2 = submission_1_df_filtered[year_columns].sum().sum()
print('Total Projected Builds between 2024-2027: ', total_proj_builds_2)



['Constrained', 'Under Construction', 'Site Complete', 'Planning Consent', 'Allocated in LDP', 'Consent Expired']
Site status
Planning Consent      64
Under Construction    62
Site Complete         23
Allocated in LDP      18
Consent Expired        9
Constrained            7
Name: count, dtype: int64
(62, 9)
Total Projected Builds yearly between 2024-2027:  Year 24/25    674
Year 25/26    343
Year 26/27    174
Year 27/28    221
dtype: int64
Total Projected Builds between 2024-2027:  1412


In [ ]:
#Create a summary statement explaining the relationship between tenure types and projected build volumes

#1. Unique tenure types and calculate total site capacity for each tenure type - without sets and dictionaries

#Unique tenure types
print('Unique values of Tenure type column: ')
print(submission_1_df['Tenure type'].value_counts())
print(submission_1_df['Tenure type'].unique().tolist())

#Group by 'Tenure type' and summarize by total 'Site capacity'
print("Groupby 'Tenure type and summarize by total 'Site capacity': ")
print(submission_1_df.groupby('Tenure type')['Site capacity'].sum().sort_values(ascending=False))

#Total projected build by Tenure type between 2024-2027 - only "CONS"

print('Toatal Projected Builds by Tenure type between 2024-2027: ')
total_proj_builds_by_tenure = submission_1_df_filtered.groupby('Tenure type')[year_columns].sum()
print(total_proj_builds_by_tenure)

total_proj_builds_by_tenure_2 = submission_1_df_filtered.groupby('Tenure type')[year_columns].sum().sum(axis=1).sort_values(ascending=False)
print(total_proj_builds_by_tenure_2)


Unique values of Tenure type column: 
Tenure type
Private                141
RSL                     26
TBC                      8
Mixed                    2
Local Authority          2
Private/RSL              2
DCC/RSL                  1
Housing Association      1
Name: count, dtype: int64
['Private', 'RSL', 'DCC/RSL', 'TBC', 'Private/RSL', 'Mixed', 'Local Authority', 'Housing Association']
Groupby 'Tenure type and summarize by total 'Site capacity': 
Tenure type
Private                3503
RSL                     840
TBC                     419
Mixed                   343
DCC/RSL                 162
Local Authority          43
Private/RSL              43
Housing Association      18
Name: Site capacity, dtype: int64
Toatal Projected Builds by Tenure type between 2024-2027: 
                     Year 24/25  Year 25/26  Year 26/27  Year 27/28
Tenure type                                                        
DCC/RSL                      26           0           0           0
Housing As

/tmp/ipython-input-3585403259.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(submission_1_df.groupby('Tenure type')['Site capacity'].sum().sort_values(ascending=False))
/tmp/ipython-input-3585403259.py:17: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  total_proj_builds_by_tenure = submission_1_df_filtered.groupby('Tenure type')[year_columns].sum()
/tmp/ipython-input-3585403259.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and sil